<h1>Real Estate Unit Price Prediction with Linear and Logistic Regression Models</h1>
<i>Original assignment: https://github.com/vesavvo/dkko/blob/211661c6bc980ee2ef852c2947cde88be65a45fc/assignments/Assignment_Linear_and_logistic_regression.md</i>

<h2>1. Business understanding</h2>
The pipeline uses data from https://archive.ics.uci.edu/dataset/477/real+estate+valuation+data+set. The dataset contains real estate prices from a <i>Taiwanese</i> city called <i>New Taipei City</i> in <i>Sindian District</i>. The goal of this pipeline is to predict real estate prices of the area using the information from the dataset. <br/> <br/>

The goal of this pipeline is to predict real estate unit prices using the features in the dataset. Real estate unit prices will be predicted using a <b>Linear Regression model</b>. In addition, a <b>Logistic Regression model</b> will be used to classify whether a given unit price is above or below the mean price. 

By using this pipeline, we aim to help residents and real estate agents in Sindian District evaluate house prices when putting homes up for sale.

---

<h2>2. Data understanding</h2>

In [106]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd

# fetch dataset 
df = fetch_ucirepo(id=477) 
df.data.original.head() #visualize the first five rows of the whole dataset

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1


<center>Table 1. A first glance at the data. The dataset contains <b>six features</b>, presumably float values, and the target variable also appears to be a float.</center>

In [109]:
df.data.original.dtypes

No                                          int64
X1 transaction date                       float64
X2 house age                              float64
X3 distance to the nearest MRT station    float64
X4 number of convenience stores             int64
X5 latitude                               float64
X6 longitude                              float64
Y house price of unit area                float64
dtype: object

<center>Table 2. Data types verified: as expected, all features are floats except 'number of convenience stores', which is an integer.</center>

<br/>

The given dataset is well documented and the variables are descriptively named, which greatly helps us interpret the data.

---

<u>The values are indicated as such in the original documentation:</u>

<b>X1:</b> The transaction date (for example, 2013.250=2013 March, 2013.500=2013 June, etc.) <br/>
<b>X2:</b> The house age (unit: year)<br/>
<b>X3:</b> The distance to the nearest MRT station (unit: meter)<br/>
<b>X4:</b> The number of convenience stores in the living circle on foot (integer)<br/>
<b>X5:</b> The geographic coordinate, latitude. (unit: degree)<br/>
<b>X6:</b> The geographic coordinate, longitude. (unit: degree)<br/> 

<br/>

The output is as follow: <br/>
<b>Y:</b> house price of unit area (10000 New Taiwan Dollar/Ping, where Ping is a local unit, 1 Ping = 3.3 meter squared)

---

In [172]:
print(df.data.original.isna().sum())
df.data.original.describe(include='all')

No                                        0
X1 transaction date                       0
X2 house age                              0
X3 distance to the nearest MRT station    0
X4 number of convenience stores           0
X5 latitude                               0
X6 longitude                              0
Y house price of unit area                0
dtype: int64


,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
count,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000
mean,207.500000,2013.148971,17.712560,1083.885689,4.094203,24.969030,121.533361,37.980193
std,119.655756,0.281967,11.392485,1262.109595,2.945562,0.012410,0.015347,13.606488
min,1.000000,2012.667000,0.000000,23.382840,0.000000,24.932070,121.473530,7.600000
25%,104.250000,2012.917000,9.025000,289.324800,1.000000,24.963000,121.528085,27.700000
50%,207.500000,2013.167000,16.100000,492.231300,4.000000,24.971100,121.538630,38.450000
75%,310.750000,2013.417000,28.150000,1454.279000,6.000000,24.977455,121.543305,46.600000
max,414.000000,2013.583000,43.800000,6488.021000,10.000000,25.014590,121.566270,117.500000


<center>Table 3 & 4. We made sure that there is no missing values and inconsistent data or data anomalies.</center>

<br/>
With <code>df.data.original.isna().sum()</code>, we get the exact count of missing values for each column. Thus, we confirmed that there are no missing values, as stated in the official documentation.

---

<h2>3. Data preparation</h2>

As we can see, there are magnitude differences in the given data, so the next step would be to normalize the values for efficient and accurate predictions. 

But before that, we have to separate the data into training and test data. We use the same split approach as was stated in the official ducumentation, so we are going to split the data 70/30, meaning 70% of data will be used for training the model and 30% for validation.

In [177]:
from sklearn.model_selection import train_test_split

X = df.data.features #dataset of only features
y = df.data.targets #dataset of only targets

#split features and targets to test and train data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)

The data has now been split into four subsets: feature and target sets for training and testing.

In [180]:
from sklearn.preprocessing import StandardScaler

# fits the scaler on training data (calculates mean and std) and transforms it
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)

# transforms test data using training parameters to prevent data leakage
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_train_scaled.columns)

X_train_scaled.head()

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude
0,-0.758469,-0.033580,0.861361,-0.390183,-0.474662,-1.353466
1,-1.646009,-0.404508,-0.462787,0.283028,-0.299716,0.249316
2,-0.758469,-0.335498,0.122613,-1.063394,0.673214,1.316976
3,0.423734,-1.198121,-0.395362,0.619633,0.266382,0.885135
4,0.129071,-0.326871,2.360495,-1.399999,-2.247226,-1.916989


<center>Table 5. A table of scaled variables</center>

<br/>

Now we have efficient magnitude on all the values and we are ready to start modeling.

---

<h2>4. Modeling</h2>

 - Valitaan koneoppimismenetelmä, opetetaan malli ja validoidaan se (validation).
 - Dokumentointiin on sisällytettävä tarkasti valittu menetelmä, käytetyt parametrit sekä mitattu mallin suorituskyky.
 - Millä perusteella tehty valinnat
 - Taulukko on hyvä viestimiseen
 - Ei tarvitse olla yksityiskohtainen, mutta selitetään miksi on päädytty lopputulokseen.

<h2>5. Evaluation</h2>

 - Arvioidaan mallin todellinen onnistuminen.
 - Selvitetään, kuinka hyvin malli vastaa sille alun perin asetettuihin liiketoimintavaatimuksiin (business requirements).<br>
 - Ei katsota mallin suorituskykyä (tämä tehdään kappaleessa 4), vaan palataan takaisin kappaleeseen 1. ja millä tavalla saavutetut tulokset vertautuu alkuperäisiin tavoitteisiin.

<h2>6. Deployment</h2>

- Malli viedään käytäntöön (deployment).
- Tähän sisältyy tulosten viestiminen ja suositusten luominen siitä, miten mallia tulisi hyödyntää käytännössä tai mitä jatkotoimenpiteitä tulisi tehdä.